<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part2_vectors/06_transfer_learning_and_embeddings/notebooks/06_transfer_learning_and_embeddings_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 06: Transfer Learning and Image Embeddings

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

A network trained on millions of images has already learned most of what it needs to see yours.
**Transfer learning** reuses that knowledge. You pass each image through a pre-trained model and
keep the vector it produces, an **image embedding**. Those vectors make a few hundred labels go a
long way, and they let you search and organize a collection before you have labeled anything.

In this notebook you will:

- use a model trained on ImageNet off the shelf, and see why its categories rarely fit research
  questions
- turn two pre-trained models (a supervised ResNet-50 and the self-supervised DINOv2) into
  embedding machines
- train a simple classifier on embeddings (a **linear probe**) and fine-tune a network, and score
  both on your 40 coded images
- draw a learning curve: how many labeled images do you need?
- search a collection by example and find near duplicates that perceptual hashes miss
- cluster food packaging into visual themes, and test how stable those themes are

**Data.** The protest and King's Day photos with your codes from module 04, and the breakfast
cereal packaging from Open Food Facts (module 02).

**Runtime.** A GPU is recommended: *Runtime → Change runtime type → T4 GPU*. On a CPU, computing
the embeddings takes a few minutes and fine-tuning (section 4) about ten.

## 0. Setup

Besides the course helpers, we need `umap-learn` (for section 7) and `imagehash` (for section 6).
`course_torch.py` contains the training loop from module 05, so we don't have to write it again.

In [ ]:
!pip install -q umap-learn imagehash

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoImageProcessor, AutoModel
from sklearn.cluster import HDBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
import imagehash
import umap

# UMAP warns that a fixed seed makes it run on one core; that is the trade-off we want
warnings.filterwarnings("ignore", message="n_jobs value")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}
BASE = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/"
for helper in ["course_data.py", "course_torch.py"]:
    if not Path(helper).exists():
        response = requests.get(BASE + helper, headers=HEADERS, timeout=30)
        response.raise_for_status()  # stop if the download failed, rather than saving an error page
        Path(helper).write_text(response.text)
import course_data
import course_torch
from course_torch import DEVICE, load_rgb

print("Running on:", DEVICE)

def show_grid(paths, labels=None, n_cols=8, size=1.6):
    """Show images in a grid, with an optional label under each."""
    paths = list(paths)
    labels = list(labels) if labels is not None else [""] * len(paths)
    n_rows = max(1, int(np.ceil(len(paths) / n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, label in zip(axes.flat, paths, labels):
        ax.imshow(load_rgb(path, 300))
        ax.set_title(str(label)[:28], fontsize=7)
    plt.tight_layout()
    plt.show()

train_table, test_table = course_data.protest_train_test()
packaging = course_data.get_packaging_corpus()

## 1. A pre-trained model, off the shelf

**ImageNet** is a collection of over a million photos sorted into 1,000 categories, from
*tabby cat* to *space shuttle*. Models trained to tell those categories apart were the
foundation of a decade of computer vision. torchvision ships several, with their trained weights
and the exact preprocessing they expect. Let's ask a ResNet-50 what it sees in a few of your test
images.

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
resnet = resnet50(weights=weights).eval().to(DEVICE)
preprocess = weights.transforms()
CATEGORIES = weights.meta["categories"]

@torch.no_grad()
def imagenet_top(paths, k=3):
    """The k most likely ImageNet categories for each image, as 'label (probability)' strings."""
    batch = torch.stack([preprocess(load_rgb(p, 512)) for p in paths]).to(DEVICE)
    probs = resnet(batch).softmax(dim=1).cpu()
    top = probs.topk(k, dim=1)
    return ["\n".join(f"{CATEGORIES[i]} ({p:.2f})" for p, i in zip(ps, idx))
            for ps, idx in zip(top.values.tolist(), top.indices.tolist())]

examples = test_table["local_path"].head(8)
show_grid(examples, imagenet_top(examples), n_cols=4, size=2.6)

The labels are often reasonable descriptions of an object in the scene, but notice what is
missing: ImageNet has no category for *protest*, *politician*, *crowd* or *celebration*. Its
categories were chosen by computer scientists for a benchmark, and they include over a hundred
dog breeds. Off-the-shelf labels can still be useful as features (is there a *flagpole*?), but
they rarely match a research question.

> **✏️ Exercise 1**
>
> Run `imagenet_top` on eight packaging photos (`packaging["local_path"]`). Which ImageNet
> categories come up? Could any of them serve as a measure in a study of food marketing, and what
> would you check first?

In [ ]:
# Your code here

## 2. From labels to embeddings

The last layer of ResNet-50 turns 2,048 numbers into 1,000 ImageNet scores. Those 2,048 numbers
are a summary of the image that the network learned while learning to tell 1,000 categories
apart, and they are useful far beyond those categories. If we remove the last layer, the network
turns every image into a vector of 2,048 numbers: an **image embedding**. Images that look alike,
in the network's sense, get vectors that point in similar directions.

We compare two models:

- **ResNet-50**, a CNN trained *with labels* on ImageNet;
- **DINOv2** (Oquab et al., 2024), a vision transformer trained *without labels* on 142 million
  images, by learning to give matching representations to different crops of the same image. Its
  small version produces 384 numbers per image. Module 08 explains how vision transformers work;
  here we just use it.

We normalize every embedding to length 1, so that the dot product of two embeddings is their
**cosine similarity**.

In [ ]:
resnet_body = resnet50(weights=weights).eval().to(DEVICE)
resnet_body.fc = nn.Identity()              # drop the ImageNet layer: the output is the embedding

@torch.no_grad()
def embed_resnet(paths, batch_size=32):
    chunks = []
    for start in range(0, len(paths), batch_size):
        batch = torch.stack([preprocess(load_rgb(p, 512)) for p in paths[start:start + batch_size]])
        chunks.append(resnet_body(batch.to(DEVICE)).cpu())
    return nn.functional.normalize(torch.cat(chunks), dim=1).numpy()

dino_processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino = AutoModel.from_pretrained("facebook/dinov2-small").eval().to(DEVICE)

@torch.no_grad()
def embed_dino(paths, batch_size=32):
    chunks = []
    for start in range(0, len(paths), batch_size):
        images = [load_rgb(p, 512) for p in paths[start:start + batch_size]]
        inputs = dino_processor(images=images, return_tensors="pt").to(DEVICE)
        chunks.append(dino(**inputs).pooler_output.cpu())
    return nn.functional.normalize(torch.cat(chunks), dim=1).numpy()

train_paths, test_paths = list(train_table["local_path"]), list(test_table["local_path"])
emb = {
    "ResNet-50": (embed_resnet(train_paths), embed_resnet(test_paths)),
    "DINOv2": (embed_dino(train_paths), embed_dino(test_paths)),
}
for name, (tr, te) in emb.items():
    print(f"{name:<10} training {tr.shape}, test {te.shape}")

## 3. A linear probe

The simplest use of embeddings: keep the pre-trained model frozen and train a plain logistic
regression on the vectors. This is called a **linear probe**. It is fast, needs no GPU once the
embeddings exist, and it is the same classifier you would use for text embeddings.

As in module 04, we train with the category labels and score against your codes.

In [ ]:
y_train = train_table["category_label"].to_numpy()
y_test = test_table["protest"].to_numpy()

probe_pred = {}
for name, (X_tr, X_te) in emb.items():
    probe = LogisticRegression(class_weight="balanced", max_iter=5000).fit(X_tr, y_train)
    probe_pred[name] = probe.predict(X_te)
    print(course_data.log_score(f"linear probe, {name} embeddings", y_test, probe_pred[name], module="06"))

course_data.show_scores()

Compare these with the random forest on handcrafted features (module 04) and the CNN trained from
scratch (module 05). The same few hundred noisy labels, but now on top of features learned from
millions of images: this is usually where the scoreboard jumps.

> **✏️ Exercise 2**
>
> Logistic regression has a setting `C` that controls how strongly it is regularized (small `C`:
> simpler model). Choose `C` for the DINOv2 probe **using the training data only**: try
> `C` in `[0.01, 0.1, 1, 10, 100]` with 5-fold cross-validation (`GridSearchCV`, scoring
> `"balanced_accuracy"`). Then evaluate the chosen model once on your codes and log it as
> `"linear probe, DINOv2, tuned C"`.
>
> Why would choosing `C` by looking at the scores on your 40 coded images be a mistake?

In [ ]:
# Your code here

## 4. Fine-tuning

A linear probe leaves the pre-trained network untouched. **Fine-tuning** goes further: it keeps
training some of the network's own weights on your task. A common middle ground is to freeze the
early layers (general edges and textures) and retrain only the last block plus a new two-class
output layer. With few labels, fine-tuning more than that tends to overfit.

We reuse the training loop from module 05 (`course_torch.train_model`), with augmentation and the
weighted loss for the unbalanced classes.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

fit_part, val_part = train_test_split(train_table, test_size=0.2, stratify=train_table["category_label"], random_state=SEED)
counts = np.bincount(fit_part["category_label"], minlength=2)
class_weights = torch.tensor(len(fit_part) / (2 * counts), dtype=torch.float32)

ImageTable, make_loader = course_torch.ImageTable, course_torch.make_loader
fit_loader = make_loader(ImageTable(fit_part["local_path"], fit_part["category_label"], train_tf, 512), shuffle=True)
val_loader = make_loader(ImageTable(val_part["local_path"], val_part["category_label"], eval_tf, 512))
test_loader = make_loader(ImageTable(test_table["local_path"], test_table["protest"], eval_tf, 512))

finetuned = resnet50(weights=weights)
for name, param in finetuned.named_parameters():
    param.requires_grad = name.startswith("layer4")      # freeze everything except the last block
finetuned.fc = nn.Linear(2048, 2)                        # a new output layer (trainable by default)
n_trainable = sum(p.numel() for p in finetuned.parameters() if p.requires_grad)
print(f"Training {n_trainable:,} of {sum(p.numel() for p in finetuned.parameters()):,} weights")

torch.manual_seed(SEED)
history = course_torch.train_model(finetuned, fit_loader, val_loader, epochs=6, lr=1e-4, class_weights=class_weights)
course_torch.plot_history(history, "ResNet-50, last block fine-tuned")

ft_pred, ft_prob = course_torch.predict(finetuned, test_loader)
print(course_data.log_score("ResNet-50, last block fine-tuned", y_test, ft_pred, module="06"))
course_data.show_scores()

Fine-tuning costs far more than a linear probe (a GPU, minutes instead of seconds, more settings
to choose) and, with a few hundred noisy labels, it often doesn't beat it. The usual advice is to
start with a linear probe on good embeddings and fine-tune only when you have more labels and
the probe has clearly hit a ceiling.

## 5. How many labels do you need?

A **learning curve** shows how performance grows with the number of labeled training images. We
train the DINOv2 probe on random subsets of increasing size, ten times per size, and score each
on your codes. The spread across the ten draws shows how much luck is involved at each size.

In [ ]:
def learning_curve(X_tr, X_te, sizes=(10, 20, 40, 80, 160), repeats=10):
    rng = np.random.default_rng(SEED)
    pos, neg = np.where(y_train == 1)[0], np.where(y_train == 0)[0]
    rows = []
    for n in list(sizes) + [len(y_train)]:
        for r in range(repeats if n < len(y_train) else 1):
            if n < len(y_train):
                n_neg = max(2, min(len(neg), round(n * len(neg) / len(y_train))))
                idx = np.concatenate([rng.choice(pos, n - n_neg, replace=False), rng.choice(neg, n_neg, replace=False)])
            else:
                idx = np.arange(len(y_train))
            probe = LogisticRegression(class_weight="balanced", max_iter=5000).fit(X_tr[idx], y_train[idx])
            rows.append({"n_labels": n, "balanced_accuracy": balanced_accuracy_score(y_test, probe.predict(X_te))})
    return pd.DataFrame(rows)

def plot_curve(curve, label):
    summary = curve.groupby("n_labels")["balanced_accuracy"].agg(["mean", "min", "max"])
    plt.plot(summary.index, summary["mean"], marker="o", label=label)
    plt.fill_between(summary.index, summary["min"], summary["max"], alpha=0.2)

curve_dino = learning_curve(*emb["DINOv2"])
plt.figure(figsize=(6, 3.5))
plot_curve(curve_dino, "DINOv2 linear probe")
plt.xscale("log")
plt.xlabel("number of labeled training images")
plt.ylabel("balanced accuracy on your codes")
plt.axhline(0.5, color="gray", linestyle=":", label="chance")
plt.legend()
plt.show()

> **✏️ Exercise 3**
>
> Draw the learning curve for the **ResNet-50** embeddings as well, in the same plot as DINOv2.
> Which embedding needs fewer labels to reach its plateau? What does the shaded band tell you about
> running a study with only 20 labeled images?

In [ ]:
# Your code here

## 6. Searching by example, and finding duplicates

Embeddings are useful before you have any labels. Because similar images get similar vectors, you
can **search a collection by example**: compute the cosine similarity between one image and all
others and look at the closest ones.

In [ ]:
all_paths = train_paths + test_paths
all_dino = np.vstack(emb["DINOv2"])
similarity = all_dino @ all_dino.T            # cosine similarity, because the vectors have length 1

def show_neighbors(query, k=6):
    order = np.argsort(-similarity[query])[: k + 1]       # the first one is the query itself
    show_grid([all_paths[i] for i in order],
              ["query"] + [f"similarity {similarity[query, i]:.2f}" for i in order[1:]], n_cols=k + 1, size=1.8)

for query in [0, 5, 10]:
    show_neighbors(query)

The same similarities reveal **near duplicates**: pairs with a very high similarity. In module 02,
perceptual hashes caught resized and recompressed copies but missed crops and mirror images.
Embeddings, trained to recognize the same scene under different crops, catch more of those.

> **✏️ Exercise 4**
>
> 1. Find all pairs of images (in `all_paths`) with a cosine similarity above 0.95 and show the
>    eight most similar pairs side by side, with their similarity and their perceptual hash
>    distance (`imagehash.phash(img1) - imagehash.phash(img2)`).
> 2. Which of these pairs would the perceptual hash threshold of 10 from module 02 have missed?
> 3. Look at pairs just below your threshold (say 0.90 to 0.95). Are they copies, or different
>    photos of the same event? How would you choose a threshold for your own corpus?

In [ ]:
# Your code here

## 7. Clustering a collection into visual themes

Embeddings also let you explore a collection before you have a codebook: group images that look
alike and see what themes emerge. The standard recipe, which you may know from BERTopic for text,
has three steps:

1. **Embed** every image (DINOv2).
2. **Reduce** the 384 dimensions to a handful with **UMAP**, which keeps images that are close in
   the embedding space close together. Clustering works better in a few dimensions.
3. **Cluster** with **HDBSCAN**, which finds dense groups of any shape, decides the number of
   clusters itself and leaves outliers unassigned (label -1).

We use the breakfast cereal packaging.

In [ ]:
package_paths = list(packaging["local_path"])
package_emb = embed_dino(package_paths)

def visual_themes(embeddings, n_components=5, min_cluster_size=8, seed=SEED):
    reduced = umap.UMAP(n_components=n_components, n_neighbors=15, min_dist=0.0, metric="cosine",
                        random_state=seed).fit_transform(embeddings)
    return HDBSCAN(min_cluster_size=min_cluster_size, copy=True).fit_predict(reduced)

themes = visual_themes(package_emb)
print(pd.Series(themes).value_counts().sort_index().rename("images per cluster (-1 = unassigned)"))

coords = umap.UMAP(n_components=2, n_neighbors=15, metric="cosine", random_state=SEED).fit_transform(package_emb)
plt.figure(figsize=(6, 4.5))
plt.scatter(coords[:, 0], coords[:, 1], c=themes, cmap="tab20", s=14)
plt.title("Cereal packaging, colored by cluster")
plt.axis("off")
plt.show()

A cluster number means nothing until you look at its images. For each cluster, we show the eight
images closest to its center.

In [ ]:
def show_clusters(labels, embeddings, paths, n=8):
    for cluster in sorted(set(labels) - {-1}):
        members = np.where(labels == cluster)[0]
        center = embeddings[members].mean(axis=0)
        closest = members[np.argsort(-(embeddings[members] @ center))][:n]
        print(f"Cluster {cluster}: {len(members)} images")
        show_grid([paths[i] for i in closest], n_cols=n, size=1.4)

show_clusters(themes, package_emb, package_paths)

> **✏️ Exercise 5**
>
> Clustering results depend on choices, just like a topic model's do.
>
> 1. Run `visual_themes` again with `min_cluster_size` set to 5 and to 15, and with a different
>    `seed`. How many clusters do you get each time, and how many images stay unassigned?
> 2. Measure how similar each result is to the original `themes` with the **adjusted Rand index**
>    (`adjusted_rand_score`: 1 means identical groupings, about 0 means no more agreement than
>    chance), using only the images that are assigned to a cluster in both runs.
> 3. Which themes are stable across runs, and which come and go?

In [ ]:
# Your code here

> **✏️ Exercise 6**
>
> Torres (2024) distinguishes clusters found in the data from **visual frames**, theory-driven ways
> of presenting an issue. Suppose your packaging clusters include one with cartoon characters and
> bright colors, one with photos of the cereal in a bowl, and one with plain bags and little imagery.
>
> 1. Can you call these clusters "frames"? Why or why not?
> 2. Describe how you would get from these clusters to a validated measure of, say, a "child appeal"
>    frame on packaging.

*Your answer here*

## Wrap-up

You have seen that:

- off-the-shelf ImageNet labels describe objects, rarely research concepts;
- removing a pre-trained model's last layer turns it into an embedding machine, and self-supervised
  models such as DINOv2 give embeddings that transfer well;
- a linear probe on good embeddings is a strong, cheap baseline, usually much better than a CNN
  trained from scratch on the same labels, and fine-tuning has to earn its extra cost;
- learning curves show how many labels you need, and how fragile results are with very few;
- embeddings support search by example and find near duplicates that perceptual hashes miss;
- clustering embeddings reveals visual themes that depend on settings, and that are candidates
  for frames, not frames.

**Next:** module 07 asks what these models learned about people, and how to audit a model for
errors that differ between groups.